In [27]:
import torch

# Simplified Self-Attention

In [28]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.22],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)

inputs.shape

torch.Size([6, 3])

In [29]:
query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0])
print(f"attn_scores_2: {attn_scores_2} ")

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

attn_scores_2: tensor([0., 0., 0., 0., 0., 0.]) 
tensor([0.9544, 1.4950, 1.4754, 0.7708, 0.7070, 1.0865])


In [30]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1398, 0.2400, 0.2353, 0.1163, 0.1091, 0.1595])
Sum: tensor(1.0000)


In [31]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1398, 0.2400, 0.2353, 0.1163, 0.1091, 0.1595])
Sum: tensor(1.0000)


In [32]:
x1 = torch.tensor([0.4, 0.1, 0.8])
x_1 = torch.tensor([0.43, 0.15, 0.89])
x2 = torch.tensor([0.5, 0.8, 0.6])
x_2 = torch.tensor([0.55, 0.87, 0.66])

print(x1)
print(x2)
print(torch.dot(x1, x2))
print(x1@x2)


print(x_1)
print(x_2)
print(x_1@x_2)


tensor([0.4000, 0.1000, 0.8000])
tensor([0.5000, 0.8000, 0.6000])
tensor(0.7600)
tensor(0.7600)
tensor([0.4300, 0.1500, 0.8900])
tensor([0.5500, 0.8700, 0.6600])
tensor(0.9544)


In [33]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4438, 0.6521, 0.5576])


In [34]:
# simplified self-attention

attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
all_context_vecs = attn_weights @ inputs

print(attn_scores.shape)
print(attn_weights.shape)
print(inputs.shape)
print(all_context_vecs)


torch.Size([6, 6])
torch.Size([6, 6])
torch.Size([6, 3])
tensor([[0.4447, 0.5933, 0.5694],
        [0.4438, 0.6521, 0.5576],
        [0.4450, 0.6502, 0.5562],
        [0.4320, 0.6289, 0.5297],
        [0.4675, 0.5910, 0.5120],
        [0.4194, 0.6509, 0.5517]])


In [35]:
print("Previous 2nd context vector:", context_vec_2)
print("  Matrix 2nd context vector:", all_context_vecs[1])

Previous 2nd context vector: tensor([0.4438, 0.6521, 0.5576])
  Matrix 2nd context vector: tensor([0.4438, 0.6521, 0.5576])


# Self-Attention

In [36]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.22],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)

x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_query


queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

print("queries.shape:", queries.shape)
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)


keys_2 = keys[1]
attn_scores_22 = query_2.dot(keys_2)

attn_scores_2 = query_2 @ keys.T

print(attn_scores_22)
print(attn_scores_2)

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

queries.shape: torch.Size([6, 2])
keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
tensor(1.8524)
tensor([1.2705, 1.8524, 1.8111, 0.9546, 0.5577, 1.5440])
tensor([0.1517, 0.2289, 0.2223, 0.1213, 0.0916, 0.1841])
tensor([0.3053, 0.8130])


In [37]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ W_query
        queries = x @ W_query
        values = x @ W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        
        return context_vec

In [38]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.22],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2987, 0.8009],
        [0.3060, 0.8200],
        [0.3057, 0.8190],
        [0.2916, 0.7809],
        [0.2911, 0.7791],
        [0.2981, 0.7990]])


In [39]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = x @ W_key
        queries = x @ W_query
        values = x @ W_key
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values

        return context_vec

In [40]:
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[0.3489, 0.8840],
        [0.3561, 0.9053],
        [0.3558, 0.9043],
        [0.3420, 0.8630],
        [0.3416, 0.8617],
        [0.3483, 0.8822]])
